# 07. Advanced RAG: Hybrid Search & Cross-Encoder Re-Ranking

Naive vector RAG fails on keyword-exact queries (product IDs, acronyms, dates) and suffers from poor chunk ordering. **Advanced RAG** fixes these issues.

### Advanced Production Components:
1. **Hybrid Search**: Fuses BM25 sparse keyword matching with dense embedding vector search via Reciprocal Rank Fusion (RRF):
   $$\text{RRF}(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$
2. **Cross-Encoder Re-Ranking**: Passes `(Query, Chunk)` pairs jointly through a cross-attention model (e.g. Cohere Rerank, BGE-Reranker) to evaluate exact semantic relevance, filtering noise.

In [ ]:
def reciprocal_rank_fusion(dense_ranks, sparse_ranks, k=60):
    rrf_scores = {}
    for rank, doc in enumerate(dense_ranks):
        rrf_scores[doc] = rrf_scores.get(doc, 0.0) + 1.0 / (k + rank + 1)
    for rank, doc in enumerate(sparse_ranks):
        rrf_scores[doc] = rrf_scores.get(doc, 0.0) + 1.0 / (k + rank + 1)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

dense_results = ["Doc_A", "Doc_B", "Doc_C"]
sparse_results = ["Doc_C", "Doc_A", "Doc_D"]

fused_ranking = reciprocal_rank_fusion(dense_results, sparse_results)
print("Hybrid Search RRF Fusion Rankings:")
for rank, (doc, score) in enumerate(fused_ranking):
    print(f"Rank {rank+1}: {doc} (RRF Score: {score:.5f})")

## 🎯 Exercises & Practice
1. What is the fundamental accuracy difference between a Bi-Encoder and a Cross-Encoder?

In [ ]:
# Exercise Solution
# Answer: Bi-Encoders embed query and doc separately (fast indexing, but zero cross-attention interactions).
# Cross-Encoders pass query and doc together through all attention layers, computing full token-to-token 
# interactions: vastly more accurate, but computationally too expensive to search millions of docs!